In [52]:
# ===== PACKAGE IMPORT ===== #
import matplotlib.pyplot as plt     
import numpy as np                  
import pyvista as pv                
import torch    
import os       
import ipywidgets
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')    

# torch.autograd.set_detect_anomaly(True)           # this was supposed to help me find the inf/ NaN errors but it didn't seem to change anything (I probably implemented it incorrectly)

In [53]:
# ===== DEFINING PARTICLES ===== #
n_part = 200

box_size = 50

r = torch.rand((n_part,3)).to(device) * box_size                                             # NEEDS UNITS AND OPTIMIZATION (multiplication increases spread of starting positions within box size)

v = torch.normal(mean=500, std = 50, size = (n_part,3)).to(device) # to be assigned later       # NEEDS UNITS AND OPTIMIZATION (configure based on box size, time step, and scale)

a = torch.zeros((n_part,3)).to(device)                                                       # NEEDS UNITS

new_a = 0       # for the very first verlet integration loop

ids = torch.arange(n_part)

ids_pairs = torch.combinations(ids, 2).to(device)       

dt = 0.0005           # NEEDS UNITS AND OPTIMIZATION
n_steps = 1000        # number of loop repeats

# for ljp -> f
sig = 1             # dimensionless
ep = 1              # dimensionless

inv_mass = 1        # dimensionless

In [54]:
# ===== POSITION - DISTANCE PAIRS ===== #
def comp_dim_pairs(r, dim):
    pairs =torch.diff(torch.combinations(r[:, dim],2).to(device), axis = 1).ravel().to(device)
    return pairs

def comp_dpairs(dx_pairs, dy_pairs, dz_pairs):
    d_pairs = torch.sqrt(dx_pairs**2 + dy_pairs**2 + dz_pairs**2).to(device)
    return d_pairs

def comp_dpairs_simp(r):
    dpairs_simp = comp_dpairs(comp_dim_pairs(r, 0), comp_dim_pairs(r, 1), comp_dim_pairs(r, 2)).to(device)
    return dpairs_simp

In [55]:
# ===== FORCE CALCULATIONS VIA NEGATIVE GRADIENT OF THE LENNARD-JONES POTENTIAL ===== #
def comp_force_pairs(r, ep, sig):
    F_r_n = ((24 * ep) / comp_dpairs_simp(r)) * (2 * ( sig / comp_dpairs_simp(r)) ** 12 - ( sig / comp_dpairs_simp(r)) ** 6).to(device)
    return F_r_n

def unit_vec_pairs(r): 
    unit_vec_pairs =torch.stack((comp_dim_pairs(r, 0)/comp_dpairs_simp(r), comp_dim_pairs(r, 1)/comp_dpairs_simp(r), comp_dim_pairs(r, 2)/comp_dpairs_simp(r)), dim = 1).to(device)
    return unit_vec_pairs

# ===== SCALAR TO VECTOR FORCE ===== #
def vec_force(r, ep, sig):
    vec_force = comp_force_pairs(r, ep, sig)[:, None] * unit_vec_pairs(r)
    return vec_force

# calculates and sums forces for each particles based on the interaction pairs it is associated with

def comp_F_sum(n_part, ids_pairs, r, ep, sig):
    F_sum = torch.zeros(n_part, 3).to(device)
    F_sum.index_add_(0, ids_pairs[:,0], vec_force(r, ep, sig))
    F_sum.index_add_(0, ids_pairs[:,1], vec_force(r, ep, sig), alpha = -1)
    return F_sum

# a hopefully working wall detection function but i have my doubts as to how effective it is
# uses equivalent of a mask to return a new tensor with the values of where the condition is met

def check_wall_coll(r, v, dim, low, high):
    out = (r[:, dim] < low) | (r[:, dim] > high)
    v[:, dim] = torch.where(out, -v[:, dim], v[:, dim])
    r[:, dim] = r[:, dim].clamp(low, high)
    return r, v

# ===== ----- ===== #
# def check_wall_coll(r, v, dim, low, high):
#     r_tmp_x_low = torch.where(r[:, dim] < low, low, r[:, dim]).to(device)
#     r_tmp_x_high = torch.where(r[:, dim] > high, high, r[:, dim]).to(device)
#     v_temp_x_low = torch.where(r[:, dim] < low, -v[:, dim], v[:, dim]).to(device)
#     v_temp_x_high = torch.where(r[:, dim] > high, -v[:, dim], v[:, dim]).to(device)
#     v[:, dim] = v_temp_x_low
#     v[:, dim] = v_temp_x_high
#     r[:, dim] = r_tmp_x_low
#     r[:, dim] = r_tmp_x_high
#     return r, v

In [56]:
# actual calculations of particles acceleration, velocity, and position
# this is supposed to run via verlet integration rather than euler, but im not sure if its set up correctly

def comp_a(n_part, ids_pairs, r, ep, sig, inv_mass):
    new_a = (comp_F_sum(n_part, ids_pairs, r, ep, sig) * inv_mass)
    return new_a

def comp_v(v, a, new_a, dt):                
    new_v = v + (a + new_a) * dt * 0.5          # note averaging accelerations over timesteps
    return new_v

def comp_r(r, v, a, dt):
    new_r = r + (v * dt) + (a * dt * dt * 0.5)      
    return new_r

def run_it_down(r, v, dt, a, n_part, ids_pairs, ep, sig, inv_mass, box_size):
    new_r = comp_r(r, v, a, dt)
    new_a = comp_a(n_part, ids_pairs, new_r, ep, sig, inv_mass)
    new_v = comp_v(v, a, new_a, dt)
    new_r, new_v = check_wall_coll(new_r, new_v, 0, 0, box_size)  # wall collisions and return unpacking
    new_r, new_v = check_wall_coll(new_r, new_v, 1, 0, box_size)
    new_r, new_v = check_wall_coll(new_r, new_v, 2, 0, box_size)
    r = new_r
    v = new_v
    a = new_a
    return r, v, a

In [57]:
# ===== PRACTICE RUN WITH TENSOR PRINTOUT ===== #

# for i in range(n_steps):
#     if i == 0:
#         a = comp_a(n_part, ids_pairs, r, ep, sig, inv_mass)
#     if i % 1 == 0:
#         r, v, a = run_it_down(r, v, dt, a, n_part, ids_pairs, ep, sig, inv_mass, box_size)
#         print("position", r)
#         print("velocity", v)
#         print("acceleration", a)

In [58]:
# ===== ANIMATION ===== 
filename = 'testingtesting123.mp4'

pl = pv.Plotter(off_screen=True)
pl.open_movie(filename)
cloud = pv.PolyData(r.cpu().numpy())
pl.add_mesh(cloud, color='black', point_size=5, render_points_as_spheres=True)
pl.show_grid(bounds=(0, box_size, 0, box_size, 0, box_size), color="grey")
pl.camera_position = [(200, 200, 200), (50, 50, 50), (0, 0, 1)]

for i in range(n_steps):
    r, v, a = run_it_down(r, v, dt, a, n_part, ids_pairs, ep, sig, inv_mass, box_size)
    cloud.points = r.cpu().numpy()
    pl.write_frame()

pl.close()

# ===== ----- ===== #
# pl = pv.Plotter(off_screen = True)
# pl.open_movie(filename)
# pl.show(auto_close = False)
# pl.camera_position = [(200, 200, 200), (50, 50, 50), (0, 0, 1)]
# np_r = r.detach().cpu().numpy() 
# ini_mesh = pv.PolyData(np_r)
# positions = pl.add_mesh(ini_mesh, color = 'black', point_size=5, render_points_as_spheres=True, reset_camera=False)
# pl.show_grid(color="grey", location = "outer")
# pl.write_frame()
# pl.clear()
# print(pv.GPUInfo())
# for i in range(n_steps):
#     pl.clear()
#     pl.camera_position = [(200, 200, 200), (50, 50, 50), (0, 0, 1)]
#     r, v, a = run_it_down(r, v, dt, a, n_part, ids_pairs, ep, sig, inv_mass, box_size) # RETURNS r, v, a
#     np_r = r.detach().cpu().numpy() 
#     # np_v = v.detach().cpu().numpy()
#     # np_a = a.detach().cpu().numpy()
#     # particles['velocity'] = np_v
#     # particles['acceleration'] = np_a
#     # velocities = particles.glyph(
#     # orient='velocity',
#     # scale = False,
#     # factor = 0.15,
#     # )
#     # accelerations = particles.glyph(
#     # orient='acceleration',
#     # scale = False,
#     # factor = 0.15,
#     # )
#     positions = pv.PolyData(np_r)
   
#     pl.add_mesh(positions, color = 'black', point_size=5, render_points_as_spheres=True, reset_camera=False)
#     # pl.add_mesh(particles['velocity'], color='blue')
#     # pl.add_mesh(particles['acceleration'], color='yellow')
#     pl.show_grid(color="grey", location = "outer")
#     pl.set_scale(xscale=1, yscale=1, zscale=1)
#     pl.render()
#     pl.write_frame()
#     time.sleep(0.01)

# pl.close()

In [59]:
# ===== EXAMPLE CODE AND GUNK IVE TRIED THAT DIDN'T WORK ===== #

# ===== example animation from pyvista documentation ===== #

# filename = 'sphere-shrinking.mp4'

# # Create a sphere with random data. Seed the rng to make it reproducible.
# rng = np.random.default_rng(seed=0)
# mesh = pv.Sphere()
# mesh.cell_data['data'] = rng.random(mesh.n_cells)

# pl = pv.Plotter()
# # Open a movie file
# pl.open_movie(filename)

# # Add initial mesh
# pl.add_mesh(mesh, scalars='data', clim=[0, 1])
# # Add outline for shrinking reference
# pl.add_mesh(mesh.outline_corners())

# pl.show(auto_close=False)  # only necessary for an off-screen movie

# # Run through each frame
# pl.write_frame()  # write initial data

# # Update scalars on each frame
# for i in range(100):
#     random_points = rng.random(mesh.points.shape)
#     mesh.points = random_points * 0.01 + mesh.points * 0.99
#     mesh.points -= mesh.points.mean(0)
#     mesh.cell_data['data'] = rng.random(mesh.n_cells)
#     pl.add_text(f'Iteration: {i}', name='time-label')
#     pl.write_frame()  # Write this frame

# # Be sure to close the plotter when finished
# pl.close()

# ===== various wall collision attempts ===== # 

# r[ r[:,0] < b , 0] = r

# r[ r[:,0] > b , 0] = b - (r-b)

# def wall_x(r, v):
#     for i in range(len(r)):
#         if r[r[i, 0] < 0, 0]: 
#             r[r[i, 0], 0] = 0
#             v[r[i, 0] < 0, 0] = -1 * v[r[i, 0] < 0, 0]
#     for j in range(len(r)):
#         if r[r[j, 0] > 10, 0]:
#             r[r[j, 0] > 10, 0] = 10
#             v[r[j, 0] > 10, 0] = -1 * v[r[i, 0] > 10, 0]
#     return r, v

# def wall_y(r, v):
#     for i in range(len(r)):
#         if r[r[i, 1] < 0, 1]:
#             r[r[i, 1] < 0, 1] = 0
#             v[r[i, 1] < 0, 1] = -1 * v[r[i, 1] < 0, 1]
#     for j in range(len(r)):
#         if r[r[j, 1] > 10, 1]:
#             r[r[j, 1] > 10, 1] = 10
#             v[r[j, 1] > 10, 1] = -1 * v[r[i, 1] > 10, 1]
#     return r, v

# def wall_z(r, v):
#     for i in range(len(r)):
#         if r[r[i, 2] < 0, 2]:
#             r[r[i, 2] < 0, 2] = 0
#             v[r[i, 2] < 0, 2] = -1 * v[r[i, 2] < 0, 0]
#     for j in range(len(r)):
#         if r[r[j, 2] > 10, 2]:
#             r[r[i, 2] > 10, 2] = 10
#             v[r[i, 2] > 10, 2] = -1 * v[r[i, 2] > 10, 0]
#     return r, v

# def check_wall_coll_y(r, v):
#     coll_id_low = torch.where(0 > r[:, 1])
#     coll_id_high = torch.where(r > 10)[:, 1]
#     for i in coll_id_low:
#         r[i, 1] = 0
#         v[i, 1] = v[i, 1] * -1
#     for j in coll_id_high:
#         r[i, 1] = 10
#         v[i, 1] = v[i, 1] * -1  
#     return r, v

# def check_wall_coll_z(r, v):
#     coll_id_low = torch.where(0 > r[:, 2])
#     coll_id_high = torch.where(r > 10)[:, 2]
#     for i in coll_id_low:
#         r[i, 2] = 0
#         v[i, 2] = v[i, 2] * -1
#     for j in coll_id_high:
#         r[i, 2] = 10
#         v[i, 2] = v[i, 2] * -1  
#     return r, v
    
# def check_wall_coll_y(r, v):
#     coll = torch.where(-5 > r[:, 1] or r[:, 1] > 5)
#     v_new= -1 * v[:, 1][coll]
#     if r[:, 1][coll] < -5:
#         r_new = r[:, 1][coll] - (r[:, 1][coll] + 5)
#     if r[:, 1][coll] > 5:
#         r_new = r[:, 1][coll] - (r[:, 1][coll] - 5)
#     return r_new, v_new

# def check_wall_coll_z(r, v):
#     coll = torch.where(-5 > r[:, 2] or r[:, 2] > 5)
#     v_new= -1 * v[:, 2][coll]
#     if r[:, 2][coll] < -5:
#         r_new = r[:, 2][coll] - (r[:, 2][coll] + 5)
#     if r[:, 2][coll] > 5:
#         r_new = r[:, 2][coll] - (r[:, 2][coll] - 5)
#     return r_new, v_new

# ===== dimension specific pair and distance calculations ===== #
# since removed after including dimension as a function argument

# def comp_dx_pairs(r):
#     x_pairs = torch.combinations(r[:,0],2).to(device)        
#     dx_pairs = torch.diff(x_pairs, axis=1).ravel()
#     return dx_pairs

# def comp_dy_pairs(r):
#     y_pairs = torch.combinations(r[:,1],2).to(device)         
#     dy_pairs = torch.diff(y_pairs, axis=1).ravel()  
#     return dy_pairs        
   
# def comp_dz_pairs(r):
#     z_pairs = torch.combinations(r[:,2],2).to(device)   
#     dz_pairs = torch.diff(z_pairs, axis=1).ravel()
#     return dz_pairs

# ===== #

# def comp_Force_x(r, ep, sig):
#     Force = comp_Force(comp_dim_pairs(r, 0), ep, sig)
#     return Force

# def comp_Force_y(r, ep, sig):
#     Force_y = comp_Force(comp_dim_pairs(r, 1), ep, sig)
#     return Force_y

# def comp_Force_z(r, ep, sig):
#     Force_z = comp_Force(comp_dim_pairs(r, 2), ep, sig)
#     return Force_z

# ===== I honestly can't remember what this was supposed to be or do ===== #

# print(ids_pairs[:,0])

# print(comp_Fr_stack(r, ep, sig))

# print(len(ids_pairs[:,0]))
# print(len(comp_Fr_stack(r, ep, sig)))

# input("input to continue")

# comp_Fr_stack(r, ep, sig)[:, ids_pairs[:,0]]

# input("input to continue")

# def comp_F_comb(ids_pairs, r, ep, sig):
#     F_comb = torch.stack((comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,0]][:,0] - comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,1]][:,0], comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,0]][:,1] - comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,1]][:,1], comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,0]][:,2] - comp_Fr_stack(r, ep, sig)[:,ids_pairs[:,1]][:,2]), dim =1 )
#     return F_comb